# Comprehensive Travel Recommender Notebook

This Jupyter Notebook documents the complete architecture, preprocessing pipeline, and mathematical formulation for the dual recommendation engine built in `app.py`. It covers both **Content-Based Filtering** and **User-Based Collaborative Filtering** using `traveler-trip-data.csv`.

---

## 1. Environment Setup and Data Pipeline

### Explanation & Objectives

The dataset requires cleaning and feature engineering before feeding into recommendation models:

* Impute missing categorical values (`Destination`, `Accommodation type`, `Transportation type`, `Traveler name`) with `'Unknown'`.


* Impute missing numerical values (`Traveler age`, `Duration (days)`) using median statistics.


* Handle cost coercions for `Accommodation cost` and `Transportation cost` to create a consolidated feature: `Total cost`.


* Categorize destinations into scenery profiles (`beach`, `city`, `mountain`) via domain lookup mapping.

In [1]:
# Imports and Data Preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv('traveler-trip-data.csv')

In [2]:
# Handle missing values
df['Destination'] = df['Destination'].fillna('Unknown')
df['Traveler name'] = df['Traveler name'].fillna('Unknown')
df['Traveler age'] = df['Traveler age'].fillna(df['Traveler age'].median())
df['Accommodation cost'] = pd.to_numeric(df['Accommodation cost'], errors='coerce').fillna(0)
df['Transportation cost'] = pd.to_numeric(df['Transportation cost'], errors='coerce').fillna(0)
df['Duration (days)'] = df['Duration (days)'].fillna(df['Duration (days)'].median())
df['Accommodation type'] = df['Accommodation type'].fillna('Unknown')
df['Transportation type'] = df['Transportation type'].fillna('Unknown')

In [3]:
df.head()

,Trip ID,Destination,Start date,End date,Duration (days),Traveler name,Traveler age,Traveler gender,Traveler nationality,Accommodation type,Accommodation cost,Transportation type,Transportation cost
0,1,"London, UK",5/1/2023,5/8/2023,7.0,John Smith,35.0,Male,American,Hotel,1200.0,Flight,600.0
1,2,"Phuket, Thailand",6/15/2023,6/20/2023,5.0,Jane Doe,28.0,Female,Canadian,Resort,800.0,Flight,500.0
2,3,"Bali, Indonesia",7/1/2023,7/8/2023,7.0,David Lee,45.0,Male,Korean,Villa,1000.0,Flight,700.0
3,4,"New York, USA",8/15/2023,8/29/2023,14.0,Sarah Johnson,29.0,Female,British,Hotel,2000.0,Flight,1000.0
4,5,"Tokyo, Japan",9/10/2023,9/17/2023,7.0,Kim Nguyen,26.0,Female,Vietnamese,Airbnb,700.0,Train,200.0


In [5]:
# Destination Category Mapping
destination_type_map = {
    'London, UK': 'city', 'Phuket, Thailand': 'beach', 'Bali, Indonesia': 'beach',
    'New York, USA': 'city', 'Tokyo, Japan': 'city', 'Paris, France': 'city',
    'Sydney, Australia': 'beach', 'Rio de Janeiro, Brazil': 'beach',
    'Amsterdam, Netherlands': 'city', 'Dubai, United Arab Emirates': 'city',
    'Cancun, Mexico': 'beach', 'Barcelona, Spain': 'beach',
    'Honolulu, Hawaii': 'beach', 'Berlin, Germany': 'city',
    'Marrakech, Morocco': 'city', 'Edinburgh, Scotland': 'city',
    'Rome': 'city', 'Bangkok': 'city', 'Hawaii': 'beach', 'Athens, Greece': 'mountain',
    'Cape Town, South Africa': 'mountain', 'Auckland, New Zealand': 'mountain'
}
df['Destination Type'] = df['Destination'].apply(lambda x: destination_type_map.get(x, 'city'))
df['Total cost'] = df['Accommodation cost'] + df['Transportation cost']

In [6]:
df.head()

,Trip ID,Destination,Start date,End date,Duration (days),Traveler name,Traveler age,Traveler gender,Traveler nationality,Accommodation type,Accommodation cost,Transportation type,Transportation cost,Destination Type,Total cost
0,1,"London, UK",5/1/2023,5/8/2023,7.0,John Smith,35.0,Male,American,Hotel,1200.0,Flight,600.0,city,1800.0
1,2,"Phuket, Thailand",6/15/2023,6/20/2023,5.0,Jane Doe,28.0,Female,Canadian,Resort,800.0,Flight,500.0,beach,1300.0
2,3,"Bali, Indonesia",7/1/2023,7/8/2023,7.0,David Lee,45.0,Male,Korean,Villa,1000.0,Flight,700.0,beach,1700.0
3,4,"New York, USA",8/15/2023,8/29/2023,14.0,Sarah Johnson,29.0,Female,British,Hotel,2000.0,Flight,1000.0,city,3000.0
4,5,"Tokyo, Japan",9/10/2023,9/17/2023,7.0,Kim Nguyen,26.0,Female,Vietnamese,Airbnb,700.0,Train,200.0,city,900.0


In [7]:
print(f"Dataset successfully processed: {len(df)} records.")
df[['Traveler name', 'Destination', 'Destination Type', 'Total cost', 'Duration (days)']].head()

Dataset successfully processed: 139 records.


,Traveler name,Destination,Destination Type,Total cost,Duration (days)
0,John Smith,"London, UK",city,1800.0,7.0
1,Jane Doe,"Phuket, Thailand",beach,1300.0,5.0
2,David Lee,"Bali, Indonesia",beach,1700.0,7.0
3,Sarah Johnson,"New York, USA",city,3000.0,14.0
4,Kim Nguyen,"Tokyo, Japan",city,900.0,7.0


---

## 2. Content-Based Filtering Engine

### Mathematical Formulation

Content-based filtering matches target criteria or item features against aggregated item profiles.

1. **Feature Transformation**:
* Continuous variables ($Total\ cost$, $Duration\ (days)$) are normalized via Z-Score Standardization:



$$z = \frac{x - \mu}{\sigma}$$


* Categorical attributes ($Destination\ Type$, $Accommodation\ type$, $Transportation\ type$) are transformed via One-Hot Encoding.




2. **Vector Similarity Computation**:
Given a user preference vector $\mathbf{u}$ and an item feature vector $\mathbf{v}$, the similarity match is computed using Cosine Similarity:


$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\Vert{}\mathbf{u}\Vert{} \Vert{}\mathbf{v}\Vert{}} = \frac{\sum_{i=1}^{n} u_i v_i}{\sqrt{\sum_{i=1}^{n} u_i^2} \sqrt{\sum_{i=1}^{n} v_i^2}}$$

### Content-Based Feature Matrix Construction

In [13]:
# Aggregate destination profiles
destination_profiles = df.groupby('Destination').agg({
    'Destination Type': 'first',
    'Total cost': 'mean',
    'Duration (days)': 'mean',
    'Accommodation type': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown',
    'Transportation type': lambda x: x.mode()[0] if not x.mode().empty else 'Unknown'
}).reset_index()

# Preprocessing Pipeline
num_features = ['Total cost', 'Duration (days)']
cat_features = ['Destination Type', 'Accommodation type', 'Transportation type']

preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), num_features),
('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)])

# Build normalized feature matrix
content_feature_matrix = preprocessor.fit_transform(destination_profiles)
print("Content Feature Matrix Shape:", content_feature_matrix.shape)

Content Feature Matrix Shape: (60, 23)


### Content-Based Recommendation Inference

In [11]:
# recommender preference function
def recommend_by_user_preference(preferred_type, max_budget, preferred_days, top_n=3):
    user_input = pd.DataFrame([{
        'Destination Type': preferred_type,
        'Total cost': max_budget,
        'Duration (days)': preferred_days,
        'Accommodation type': 'Unknown',
        'Transportation type': 'Unknown'
    }])

    user_vector = preprocessor.transform(user_input)
    sim_scores = cosine_similarity(user_vector, content_feature_matrix).flatten()

    res = destination_profiles.copy()
    res['Match Score (%)'] = (sim_scores * 100).round(1)

    # Apply dynamic budget boundary (within 20% tolerance)
    filtered = res[res['Total cost'] <= max_budget * 1.2]
    if filtered.empty:
        filtered = res

    return filtered.sort_values(by='Match Score (%)', ascending=False).head(top_n)

# Example Execution
print("--- Content-Based Test Run ---")
cb_recommendations = recommend_by_user_preference(preferred_type='beach', max_budget=2000, preferred_days=7)
cb_recommendations[['Destination', 'Destination Type', 'Total cost', 'Duration (days)', 'Match Score (%)']]

--- Content-Based Test Run ---


,Destination,Destination Type,Total cost,Duration (days),Match Score (%)
58,Unknown,city,0.0,7.00,54.4
42,"Phuket, Thailand",beach,1300.0,5.00,41.9
15,"Cancun, Mexico",beach,1600.0,7.25,38.3


---

## 3. Collaborative Filtering Engine

### Mathematical Formulation

User-Based Collaborative Filtering recommends items based on peer neighborhood patterns rather than metadata properties.

1. **Implicit Engagement Score Generation**:
Since the original dataset lacks explicit 1–5 numerical ratings, an implicit engagement metric is generated based on daily expenditure ($CPD$):


$$CPD = \frac{Total\ cost}{\max(Duration\ (days), 1)}$$


This is scaled onto a normalized $1.0 - 5.0$ rating range:


$$\text{Rating} = 1 + 4 \times \left( \frac{CPD - CPD_{\min}}{CPD_{\max} - CPD_{\min} + \epsilon} \right)$$


2. **User Neighborhood Matrix**:
A User-Item Matrix $R \in \mathbb{R}^{U \times I}$ is constructed. User-user similarity matrix $S$ is generated using Cosine Similarity between user rating vectors $\mathbf{r}_a$ and $\mathbf{r}_b$:


$$\text{Sim}(a, b) = \frac{\mathbf{r}_a \cdot \mathbf{r}_b}{\Vert{}\mathbf{r}_a\Vert{} \Vert{}\mathbf{r}_b\Vert{}}$$


3. **Prediction Aggregation**:
For target user $u$ and unvisited destination $i$, the predicted affinity score $\hat{r}_{u,i}$ is calculated as a similarity-weighted average over common peers $v \in N$:


$$\hat{r}_{u,i} = \frac{\sum_{v \in N} \text{Sim}(u, v) \cdot r_{v,i}}{\sum_{v \in N} \text{Sim}(u, v)}$$

In [ ]:
# Cell 4: Collaborative Filtering Matrix Setup
df_collab = df.copy()

# Compute Cost Per Day (CPD)
df_collab['Cost_Per_Day'] = df_collab['Total cost'] / np.maximum(df_collab['Duration (days)'], 1)

min_cpd = df_collab['Cost_Per_Day'].min()
max_cpd = df_collab['Cost_Per_Day'].max()
df_collab['Implicit_Rating'] = 1 + 4 * (df_collab['Cost_Per_Day'] - min_cpd) / (max_cpd - min_cpd + 1e-5)

# Build User-Item Interaction Matrix
user_item_matrix = df_collab.pivot_table(
    index='Traveler name',
    columns='Destination',
    values='Implicit_Rating',
    aggfunc='mean'
)

# Compute User Similarity DataFrame
user_item_filled = user_item_matrix.fillna(0)
user_similarity = cosine_similarity(user_item_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print(f"User-Item Matrix shape: {user_item_matrix.shape}")

User-Item Matrix shape: (110, 60)


In [ ]:
# Cell 5: Collaborative Recommendation Inference
def recommend_collaborative(selected_traveler, top_n=3):
    if selected_traveler not in user_item_matrix.index:
        raise KeyError(f"Traveler '{selected_traveler}' not found in database.")

    # Sort similar users
    similar_users = user_similarity_df[selected_traveler].sort_values(ascending=False).drop(selected_traveler)

    # Identify unvisited destinations
    user_ratings = user_item_matrix.loc[selected_traveler]
    unvisited = user_ratings[user_ratings.isna()].index

    predicted_ratings = {}
    for dest in unvisited:
        other_ratings = user_item_matrix[dest].dropna()
        common = similar_users.index.intersection(other_ratings.index)

        if len(common) > 0:
            sims = similar_users.loc[common]
            rats = other_ratings.loc[common]
            if sims.sum() > 0:
                pred = np.dot(sims, rats) / sims.sum()
                predicted_ratings[dest] = round(pred, 2)

    rec_df = pd.DataFrame(list(predicted_ratings.items()), columns=['Destination', 'Predicted Affinity Score'])
    top_recs = rec_df.sort_values(by='Predicted Affinity Score', ascending=False).head(top_n)
    top_peers = similar_users.head(3)

    return top_recs, top_peers

# Example Execution
test_user = user_item_matrix.index[0]
print(f"--- Collaborative Filtering Test Run for Traveler: '{test_user}' ---")

cf_recs, top_peers = recommend_collaborative(test_user, top_n=3)
print("\nTop Recommended Destinations:")
print(cf_recs.to_string(index=False))

print("\nTop Similar Travelers:")
for peer, sim in top_peers.items():
    print(f"Traveler: {peer:<25} | Cosine Similarity: {sim:.3f}")

--- Collaborative Filtering Test Run for Traveler: 'Adam Lee' ---

Top Recommended Destinations:
       Destination  Predicted Affinity Score
New York City, USA                      5.00
             Tokyo                      1.19
          New York                      1.00

Top Similar Travelers:
Traveler: Sarah Lee                 | Cosine Similarity: 0.955
Traveler: Emily Davis               | Cosine Similarity: 0.274
Traveler: Alex Ng                   | Cosine Similarity: 0.000


---

## 4. Algorithm Comparison

| Attribute | Content-Based Engine| Collaborative Engine|
| --- | --- | --- |
| **Input Data** | Descriptive item features (Budget, Duration, Scenery)| User travel history interaction matrices|
| **Cold Start** | Works for new/unvisited destinations| Requires prior interaction/travel data for users/items|
| **Recommendation Basis** | Vector distance between preferences and destination profiles| Behavioral similarity with other travelers|
| **Primary Metric** | Scaled Cosine Similarity match percentage (%)| Predicted Affinity Score (1.0 to 5.0 scale)|